# Classification — Part B (Day 1 Scaffold)
## 23CSE301 ML Capstone · Review 2
### Person 1 | Track: Classification Part B
---
**Problem Statement:**  
Predict industrial machine failure (binary: `Machine failure` = 0 or 1) from sensor readings
using the AI4I 2020 Predictive Maintenance Dataset.  
This notebook extends Review 1's Part A classifiers with five additional Part B algorithms.

**Dataset:** ai4i2020.csv — 10,000 rows × 14 columns  
**Target column:** `Machine failure` (binary: 0 = No failure, 1 = Failure)

**Day 1 Scope:**
- Environment setup and dataset loading
- Feature engineering (reuse from Review 1)
- Preprocessing pipeline
- Part B Classifiers 6, 7, 8: Random Forest, AdaBoost, Gradient Boosting
- Initial metric computation

> **ACADEMIC INTEGRITY NOTE:** Code cells are scaffolding assistance.  
> All analytical observations, conclusions, and interpretations in Markdown cells  
> must be written by the student from actual output inspection.


## 1. Environment Setup

In [ ]:
# Cell 1.1 — Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Part B classifiers (Day 1)
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)

# Shared evaluation utilities (same directory)
# eval_utils.py is in the same directory as this notebook
# (Review2/notebooks/classification/eval_utils.py)
sys.path.insert(0, os.getcwd())
from eval_utils import (
    compute_classification_metrics,
    build_results_table,
    plot_confusion_matrix,
    plot_roc_curves,
    plot_metric_barplot,
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__}  |  Pandas: {pd.__version__}")


## 2. Dataset Loading

In [ ]:
# Cell 2.1 — Load dataset
# Relative path: works when notebook is in Review2/notebooks/ and CSV is in Review2/data/
DATA_PATH = os.path.join('..', '..', 'data', 'ai4i2020.csv')  # Review2/data/ai4i2020.csv

# Fallback for Day 1 handoff (CSV in same folder's data/)
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('..', '..', 'data', 'ai4i2020.csv')  # canonical fallback

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(3)


In [ ]:
# Cell 2.2 — Quick sanity check
print("=== Missing Values ===")
print(df.isnull().sum())
print()
print("=== Target Distribution ===")
print(df['Machine failure'].value_counts())
print(f"\nFailure rate: {df['Machine failure'].mean()*100:.2f}%")


## 3. Feature Engineering and Preprocessing

In [ ]:
# Cell 3.1 — Feature definition
# Identifier columns to drop (not predictive features)
DROP_COLS = ['UDI', 'Product ID']

# Target column
TARGET = 'Machine failure'

# Encode 'Type' (L/M/H) as ordinal — L=0, M=1, H=2 (consistent with Review 1)
type_map = {'L': 0, 'M': 1, 'H': 2}
df['Type_encoded'] = df['Type'].map(type_map)

# Feature columns for classification (exclude identifiers, target, and failure sub-types)
EXCLUDE_FROM_FEATURES = DROP_COLS + [TARGET, 'Type', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
FEATURES = [col for col in df.columns if col not in EXCLUDE_FROM_FEATURES]

print("Features:", FEATURES)
print("Target:", TARGET)


In [ ]:
# Cell 3.2 — Train/test split (80:20, stratified)
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape}  |  Test set: {X_test.shape}")
print(f"Train failure rate: {y_train.mean()*100:.2f}%")
print(f"Test  failure rate: {y_test.mean()*100:.2f}%")


In [ ]:
# Cell 3.3 — Scaling (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling applied (fitted on training set only — no data leakage).")


## 4. Part B Classifiers — Day 1

### 4.1 Classifier 6: Random Forest

In [ ]:
# Cell 4.1 — Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf  = rf.predict_proba(X_test_scaled)

metrics_rf = compute_classification_metrics(
    "Random Forest", y_test, y_pred_rf, y_prob_rf
)
print(metrics_rf)


In [ ]:
# Cell 4.1b — Random Forest: confusion matrix
plot_confusion_matrix(y_test, y_pred_rf, "Random Forest",
                      class_names=['No Failure', 'Failure'])


### 4.2 Classifier 7: AdaBoost

> **STUDENT TASK (Day 1 observation):**  
> After running the cells, inspect the AdaBoost confusion matrix and metrics.  
> Write 2–3 sentences noting: how many failures were correctly detected,  
> how many were missed (false negatives), and what that means for a real maintenance system.  
> Do NOT copy a generic description — refer to the actual numbers you see.


In [ ]:
# Cell 4.2 — AdaBoost Classifier
ada = AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE)
ada.fit(X_train_scaled, y_train)

y_pred_ada = ada.predict(X_test_scaled)
y_prob_ada  = ada.predict_proba(X_test_scaled)

metrics_ada = compute_classification_metrics(
    "AdaBoost", y_test, y_pred_ada, y_prob_ada
)
print(metrics_ada)


In [ ]:
# Cell 4.2b — AdaBoost: confusion matrix
plot_confusion_matrix(y_test, y_pred_ada, "AdaBoost",
                      class_names=['No Failure', 'Failure'])


### 4.3 Classifier 8: Gradient Boosting

In [ ]:
# Cell 4.3 — Gradient Boosting Classifier
gb = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
gb.fit(X_train_scaled, y_train)

y_pred_gb = gb.predict(X_test_scaled)
y_prob_gb  = gb.predict_proba(X_test_scaled)

metrics_gb = compute_classification_metrics(
    "Gradient Boosting", y_test, y_pred_gb, y_prob_gb
)
print(metrics_gb)


In [ ]:
# Cell 4.3b — Gradient Boosting: confusion matrix
plot_confusion_matrix(y_test, y_pred_gb, "Gradient Boosting",
                      class_names=['No Failure', 'Failure'])


## 5. Day 1 Partial Results

In [ ]:
# Cell 5.1 — Partial results table (Day 1 classifiers only)
day1_results = [metrics_rf, metrics_ada, metrics_gb]
day1_table = build_results_table(day1_results)
print("=== Day 1 Part B Results (Classifiers 6–8) ===")
day1_table


## 6. Day 1 Verification Checklist

Run this cell to confirm all Day 1 requirements are met.


In [ ]:
# Cell 6.1 — Verification checks
checks = {
    "Dataset loaded": df.shape == (10000, 14),
    "Train/test split is 80:20": len(X_train) == 8000 and len(X_test) == 2000,
    "Scaler fitted on train only": True,  # Enforced by code structure
    "Random Forest trained": 'y_pred_rf' in dir(),
    "AdaBoost trained": 'y_pred_ada' in dir(),
    "Gradient Boosting trained": 'y_pred_gb' in dir(),
    "Metrics computed for all 3": len(day1_results) == 3,
}
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    print(f"[{status}] {check}")


---
## Day 1 Complete — Handoff Notes

**Next (Day 2):** Person 1 will add Classifiers 9 and 10 (Bagging + MLP) to this notebook.

**Student must do before Day 2:**
1. Run all cells top-to-bottom and fix any errors.
2. Write your own observation in Section 4.2 (AdaBoost) and in any blank Markdown cells.
3. Note which metric is most important for a maintenance failure system and why.
   (Hint: think about the cost of missing a real failure vs. raising a false alarm.)
